In [1]:
import pandas as pd 
import os

### Setting folder path

In [2]:
folder_path=r"C:\Users\lokes\Desktop\Data Analytics Projects\Customer Behavior & Revenue Intelligence"

### Extracting csv files

## Data Profiling

In [3]:
csv_files=[file for file in os.listdir(folder_path) if file.endswith('.csv')]

In [4]:
for file in csv_files:
    print("-"*80)
    print(f"DATASET:{file}")
    print("-"*80)

    df=pd.read_csv(os.path.join(folder_path,file))

    ## getting shape(rows,colums)
    
    print("\nShape:")
    print(df.shape)

    ## column list
    print("\nColumns:")
    print(df.columns.tolist())

    ##validating data types
    print("\nData Types:")
    print(df.dtypes)

    ## checking for nulls
    print("\nMissing Values:")
    print(df.isnull().sum())

    ## checking for duplicate rows
    print("\nDuplicate Rows:")
    print(df.duplicated().sum())    

    # print("\nFirst 5 Rows:")
    print(df.head())
    

--------------------------------------------------------------------------------
DATASET:customers_clean.csv
--------------------------------------------------------------------------------

Shape:
(99441, 5)

Columns:
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

Data Types:
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object

Missing Values:
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

Duplicate Rows:
0
                        customer_id                customer_unique_id  \
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
2  4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229

## Data Cleaning

In [5]:
# Loading datasets

In [6]:
customers=pd.read_csv(os.path.join(folder_path,"olist_customers_dataset.csv"))
orders=pd.read_csv(os.path.join(folder_path,"olist_orders_dataset.csv"))
order_items=pd.read_csv(os.path.join(folder_path,"olist_order_items_dataset.csv"))
geolocation=pd.read_csv(os.path.join(folder_path,"olist_geolocation_dataset.csv"))
payments=pd.read_csv(os.path.join(folder_path,"olist_order_payments_dataset.csv"))
products=pd.read_csv(os.path.join(folder_path,"olist_products_dataset.csv"))
reviews=pd.read_csv(os.path.join(folder_path,"olist_order_reviews_dataset.csv"))
sellers=pd.read_csv(os.path.join(folder_path,"olist_sellers_dataset.csv"))
product_category=pd.read_csv(os.path.join(folder_path,"product_category_name_translation.csv"))

In [7]:
# Datatypes standardization
customers['customer_zip_code_prefix']= (customers['customer_zip_code_prefix'].astype(str))
geolocation['geolocation_zip_code_prefix']= (geolocation['geolocation_zip_code_prefix'].astype(str))
sellers['seller_zip_code_prefix']= (sellers['seller_zip_code_prefix'].astype(str))
print(f"customers :\n{customers.dtypes}")
print(f"geolocation: \n{geolocation.dtypes}")
print(f"sellers: \n{sellers.dtypes}")



customers :
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix    object
customer_city               object
customer_state              object
dtype: object
geolocation: 
geolocation_zip_code_prefix     object
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                object
geolocation_state               object
dtype: object
sellers: 
seller_id                 object
seller_zip_code_prefix    object
seller_city               object
seller_state              object
dtype: object


In [8]:
# Data standardization for dates columns

In [9]:

order_date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in order_date_cols:
    orders[col] = pd.to_datetime(
        orders[col],
        dayfirst=True,
        errors="coerce"
    )

review_date_col=[
    "review_creation_date",
    "review_answer_timestamp"
]
for col in review_date_col:
    reviews[col]=pd.to_datetime(reviews[col],dayfirst=True)

order_items['shipping_limit_date']=pd.to_datetime(order_items['shipping_limit_date'],dayfirst=True)


In [10]:
orders.info()
reviews.info()
order_items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  object        
 1   customer_id                    99441 non-null  object        
 2   order_status                   99441 non-null  object        
 3   order_purchase_timestamp       99441 non-null  datetime64[ns]
 4   order_approved_at              99281 non-null  datetime64[ns]
 5   order_delivered_carrier_date   97658 non-null  datetime64[ns]
 6   order_delivered_customer_date  96476 non-null  datetime64[ns]
 7   order_estimated_delivery_date  99441 non-null  datetime64[ns]
dtypes: datetime64[ns](5), object(3)
memory usage: 6.1+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype    

### Identifying missing values

In [11]:
orders.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [12]:
orders['order_status'].value_counts()

#order status count where date is null
orders[
    orders["order_delivered_customer_date"].isnull()]["order_status"].value_counts()

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

In [13]:
orders=orders[
    ~(orders["order_status"] == "delivered") &
    (orders["order_delivered_customer_date"].isnull())
]

In [14]:
orders [
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_customer_date"].isnull())
]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date


In [15]:
products['product_category_name']=products['product_category_name'].fillna("Unknown")

In [16]:
metadata_cols = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty"
]

for col in metadata_cols:
    products[col] = products[col].fillna(
        products[col].median()
    )

In [17]:
dimension_cols = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

for col in dimension_cols:
    products[col] = products[col].fillna(
        products[col].median()
    )

In [18]:
reviews["review_comment_title"] = (
    reviews["review_comment_title"]
    .fillna("No Title")
)

reviews["review_comment_message"] = (
    reviews["review_comment_message"]
    .fillna("No Comment")
)

In [19]:
reviews.isnull().sum()

review_id                  0
order_id                   0
review_score               0
review_comment_title       0
review_comment_message     0
review_creation_date       0
review_answer_timestamp    0
dtype: int64

In [20]:
geolocation.duplicated().sum()

np.int64(261836)

In [21]:
geolocation[geolocation.duplicated()].head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
15,1046,-23.546081,-46.644820,sao paulo,SP
44,1046,-23.546081,-46.644820,sao paulo,SP
65,1046,-23.546081,-46.644820,sao paulo,SP
66,1009,-23.546935,-46.636588,sao paulo,SP
67,1046,-23.546081,-46.644820,sao paulo,SP


In [22]:
geolocation = geolocation.drop_duplicates()

In [23]:
geolocation.duplicated().sum()

np.int64(0)

In [24]:
geolocation.shape

(738327, 5)

# Feature Engineering

### Delivary Days

In [25]:
orders=pd.read_csv(os.path.join(folder_path,"olist_orders_dataset.csv"))


In [26]:
order_date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in order_date_cols:
    orders[col] = pd.to_datetime(
        orders[col],
        dayfirst=True,
        errors="coerce"
    )

In [27]:
orders = orders[
    ~(
        (orders["order_status"] == "delivered") &
        (orders["order_delivered_customer_date"].isnull())
    )
]

In [28]:
orders.shape

(99433, 8)

In [29]:
orders["order_delivered_customer_date"].notnull().sum()

np.int64(96476)

In [30]:
orders = orders.copy()

In [31]:
orders["delivery_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_purchase_timestamp"]
).dt.days

In [32]:
orders["delivery_days"].describe()

count    96476.000000
mean        12.094511
std          9.551801
min          0.000000
25%          6.000000
50%         10.000000
75%         15.000000
max        209.000000
Name: delivery_days, dtype: float64

In [33]:
orders["delivery_days"].head()

0     8.0
1    13.0
2     9.0
3    13.0
4     2.0
Name: delivery_days, dtype: float64

### Delivary Delays

In [34]:
orders["delivery_delay_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_estimated_delivery_date"]
).dt.days

In [35]:
orders["delivery_delay_days"].describe()

count    96476.000000
mean       -11.876881
std         10.183854
min       -147.000000
25%        -17.000000
50%        -12.000000
75%         -7.000000
max        188.000000
Name: delivery_delay_days, dtype: float64

### Late Delivary Flag

In [36]:
orders["late_delivery_flag"] = (
    orders["delivery_delay_days"] > 0
).astype(int)

In [37]:
orders["late_delivery_flag"].value_counts()

late_delivery_flag
0    92898
1     6535
Name: count, dtype: int64

### Purchase Year

In [38]:
orders["purchase_year"] = (
    orders["order_purchase_timestamp"]
    .dt.year
)

In [39]:
orders["purchase_year"].value_counts()

purchase_year
2018    54005
2017    45099
2016      329
Name: count, dtype: int64

In [40]:
orders["purchase_month"] = (
    orders["order_purchase_timestamp"]
    .dt.month_name()
)

In [41]:
orders["purchase_month"].value_counts()

purchase_month
August       10843
May          10572
July         10315
March         9893
June          9409
April         9343
February      8508
January       8069
November      7543
December      5674
October       4959
September     4305
Name: count, dtype: int64

## Total Item Value

In [42]:
order_items["total_item_value"] = (
    order_items["price"]
    + order_items["freight_value"]
)

In [43]:
order_items[
    ["price",
     "freight_value",
     "total_item_value"]
].head()

,price,freight_value,total_item_value
0,58.90,13.29,72.19
1,239.90,19.93,259.83
2,199.00,17.87,216.87
3,12.99,12.79,25.78
4,199.90,18.14,218.04


## Freight Ratio

In [44]:
order_items["freight_ratio"] = (
    (order_items["freight_value"]
    / order_items["price"]
    ) * 100
).round(2)

In [45]:
order_items[
    ["price",
     "freight_value",
     "freight_ratio"]
].head()

,price,freight_value,freight_ratio
0,58.90,13.29,22.56
1,239.90,19.93,8.31
2,199.00,17.87,8.98
3,12.99,12.79,98.46
4,199.90,18.14,9.07


In [46]:
order_items["freight_ratio"].describe()

count    112650.000000
mean         32.086322
std          34.989360
min           0.000000
25%          13.400000
50%          23.135000
75%          39.300000
max        2623.530000
Name: freight_ratio, dtype: float64

In [47]:
order_value = (
    order_items
    .groupby("order_id")["total_item_value"]
    .sum()
    .reset_index()
)

In [48]:
order_value.rename(
    columns={
        "total_item_value":
        "order_total_value"
    },
    inplace=True
)

In [49]:
order_value.head()

,order_id,order_total_value
0,00010242fe8c5a6d1ba2dd792cb16214,72.19
1,00018f77f2f0320c557190d7a144bdd3,259.83
2,000229ec398224ef6ca0657da4fc703e,216.87
3,00024acbcdf0a6daa1e931b038114c75,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04


## Number of items per order

In [50]:
items_per_order = (
    order_items
    .groupby("order_id")
    .size()
    .reset_index(name="items_count")
)

In [51]:
items_per_order.sample(10)

,order_id,items_count
69038,b379b3d63e1948d3692990c4e0a5adc8,1
17975,2ef6fc47bb9f3e0ac4a943d408fa123f,1
66606,ad65e95a693ee0523cd2eafe78406feb,1
590,01946da9e2db8bab61d79296c037ee1f,1
71816,ba81383846b0b6c97b5160f986fa3707,1
816,022b1f3510a95b396431fec95f5de236,1
44733,741be01316057f2056a6cb0f84c8ceb6,1
35975,5db2ef66c9c43c3ede53f72e6d10760c,1
60926,9ee8c4e95442d267ec52b5c452928131,1
44669,73f38e238737fb52c78ba2002c2e9213,1


In [52]:
items_per_order["items_count"].describe()

count    98666.000000
mean         1.141731
std          0.538452
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         21.000000
Name: items_count, dtype: float64

## Avg Product price per order

In [53]:
avg_price_order = (
    order_items
    .groupby("order_id")["price"]
    .mean()
    .reset_index()
)

avg_price_order.rename(
    columns={
        "price":
        "avg_product_price"
    },
    inplace=True
)

In [54]:
avg_price_order.head()

,order_id,avg_product_price
0,00010242fe8c5a6d1ba2dd792cb16214,58.90
1,00018f77f2f0320c557190d7a144bdd3,239.90
2,000229ec398224ef6ca0657da4fc703e,199.00
3,00024acbcdf0a6daa1e931b038114c75,12.99
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90


In [55]:
avg_price_order.describe()

,avg_product_price
count,98666.000000
mean,125.919255
std,190.985636
min,0.850000
25%,41.990000
50%,79.000000
75%,139.900000
max,6735.000000


## High Value Order

In [56]:
order_value["high_value_order"] = (
    order_value["order_total_value"]
    > order_value[
        "order_total_value"
    ].median()
).astype(int)

In [57]:
order_value.head()

,order_id,order_total_value,high_value_order
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,0
1,00018f77f2f0320c557190d7a144bdd3,259.83,1
2,000229ec398224ef6ca0657da4fc703e,216.87,1
3,00024acbcdf0a6daa1e931b038114c75,25.78,0
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,1


### Payment Behavior

In [58]:
payments["used_installment"] = (
    payments["payment_installments"]
    > 1
).astype(int)

In [59]:
payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value,used_installment
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33,1
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39,0
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71,0
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78,1
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45,1


In [60]:
payments['used_installment'].value_counts()

used_installment
0    52548
1    51338
Name: count, dtype: int64

### Review Sentiments

In [61]:
reviews["review_sentiment"] = (
    reviews["review_score"]
    .apply(
        lambda x:
        "Positive"
        if x >= 4
        else (
            "Neutral"
            if x == 3
            else "Negative"
        )
    )
)

In [62]:
reviews["review_sentiment"].value_counts()

review_sentiment
Positive    76470
Negative    14575
Neutral      8179
Name: count, dtype: int64

# **Revenue Intelligence**

## Revenue by year

## Top 10 States by Revenue

In [63]:
state_mapping = {
    "AC": "Acre",
    "AL": "Alagoas",
    "AP": "Amapá",
    "AM": "Amazonas",
    "BA": "Bahia",
    "CE": "Ceará",
    "DF": "Distrito Federal",
    "ES": "Espírito Santo",
    "GO": "Goiás",
    "MA": "Maranhão",
    "MT": "Mato Grosso",
    "MS": "Mato Grosso do Sul",
    "MG": "Minas Gerais",
    "PA": "Pará",
    "PB": "Paraíba",
    "PR": "Paraná",
    "PE": "Pernambuco",
    "PI": "Piauí",
    "RJ": "Rio de Janeiro",
    "RN": "Rio Grande do Norte",
    "RS": "Rio Grande do Sul",
    "RO": "Rondônia",
    "RR": "Roraima",
    "SC": "Santa Catarina",
    "SP": "São Paulo",
    "SE": "Sergipe",
    "TO": "Tocantins"
}

## High value order

In [64]:
threshold = order_value[
    "order_total_value"
].quantile(0.75)

(
    order_value[
        "order_total_value"
    ] > threshold
).mean() * 100

np.float64(24.995439158372694)

# **Customer Behavior KPI**

## Repeate customer Rate

In [75]:
items_per_order[
    "items_count"
].mean()

np.float64(1.1417306873695092)

## Customer Satisfaction KPI

In [ ]:
promoters = (
    master_df["review_score"] >= 4
).mean() * 100

print(promoters)

detractors = (
    master_df["review_score"] <= 2
).mean() * 100

print(detractors)

## Total Revenue

In [66]:
payments["payment_value"].sum()

np.float64(16008872.12)

## Total Orders

In [67]:
orders["order_id"].nunique()

99433

## Total customers

In [68]:

customers["customer_unique_id"].nunique()

96096

## AOV


In [69]:
payments["payment_value"].sum() / orders["order_id"].nunique()

np.float64(161.00160027355102)

## Avg delivery days

In [70]:
orders["delivery_days"].mean()

np.float64(12.094510551847092)

## Late Delivery %

In [71]:
orders["late_delivery_flag"].mean()*100

np.float64(6.572264741081934)

## AVG Review Score

In [77]:
avg_review_score = reviews["review_score"].mean()
print(round(avg_review_score,2))

4.09


## Revenue Loss Due to Failed Orders

In [82]:
failed_orders = orders.loc[
    orders["order_status"].isin(
        ["canceled", "unavailable"]
    ),
    "order_id"
]

revenue_loss = (
    order_items.loc[
        order_items["order_id"].isin(failed_orders),
        "total_item_value"
    ]
    .sum()
)

print(f"{revenue_loss:,.2f}")

108,026.21


## Late Delivery Impact On Customer Satisfaction

In [84]:
delivery_review_df = orders.merge(
    reviews,
    on="order_id",
    how="inner"
)

In [86]:
impact = (
    delivery_review_df
    .groupby('late_delivery_flag')
    ['review_score']
    .mean()
    .reset_index()
)

impact['Delivery Status'] = impact[
    'late_delivery_flag'
].map({
    0: 'On-Time',
    1: 'Late'
})

impact

,late_delivery_flag,review_score,Delivery Status
0,0,4.211764,On-Time
1,1,2.271139,Late


## Total customers Experiencing late and ontime delivery

In [87]:
orders.groupby(
    'late_delivery_flag'
)['customer_id'].nunique()

late_delivery_flag
0    92898
1     6535
Name: customer_id, dtype: int64

## Top 10 categories by revenue


In [72]:
products_en = products.merge(
    product_category,
    on='product_category_name',
    how='left'
)

top_10_categories = (
    order_items
    .merge(
        products_en[
            [
                'product_id',
                'product_category_name_english'
            ]
        ],
        on='product_id',
        how='left'
    )
    .merge(
        payments[
            [
                'order_id',
                'payment_value'
            ]
        ],
        on='order_id',
        how='left'
    )
    .groupby(
        'product_category_name_english'
    )['payment_value']
    .sum()
    .sort_values(
        ascending=False
    )
    .head(10)
)

top_10_categories

product_category_name_english
bed_bath_table           1712553.67
health_beauty            1657373.12
computers_accessories    1585330.45
furniture_decor          1430176.39
watches_gifts            1429216.68
sports_leisure           1392127.56
housewares               1094758.13
auto                      852294.33
garden_tools              838280.75
cool_stuff                779698.00
Name: payment_value, dtype: float64

In [73]:
revenue_trend = (
    orders[
        [
            'order_id',
            'order_purchase_timestamp'
        ]
    ]
    .merge(
        payments[
            [
                'order_id',
                'payment_value'
            ]
        ],
        on='order_id',
        how='left'
    )
)

revenue_trend['year_month'] = (
    revenue_trend[
        'order_purchase_timestamp'
    ]
    .dt.to_period('M')
)

monthly_revenue = (
    revenue_trend
    .groupby('year_month')[
        'payment_value'
    ]
    .sum()
    .reset_index()
)

monthly_revenue

,year_month,payment_value
0,2016-09,252.24
1,2016-10,59090.48
2,2016-12,19.62
3,2017-01,138488.04
4,2017-02,291908.01
5,2017-03,449863.60
6,2017-04,417788.03
7,2017-05,592724.82
8,2017-06,511276.38
9,2017-07,592382.92


# Loading back datasets

In [74]:
# 1. Products + category translation merge
products_final = products.merge(
    product_category,
    on="product_category_name",
    how="left"
)

# Fill missing translated categories
products_final[
    "product_category_name_english"
] = products_final[
    "product_category_name_english"
].fillna("Unknown Category")


# 2. Save cleaned tables
orders.to_csv(
    "orders_clean.csv",
    index=False
)

customers.to_csv(
    "customers_clean.csv",
    index=False
)

order_items.to_csv(
    "order_items_clean.csv",
    index=False
)

payments.to_csv(
    "payments_clean.csv",
    index=False
)

reviews.to_csv(
    "reviews_clean.csv",
    index=False
)

products_final.to_csv(
    "products_clean.csv",
    index=False
)